In [1]:
import pandas as pd
import numpy as np
from datetime import datetime as dt,timedelta
from datetime import datetime,date
import matplotlib.pyplot as plt
import seaborn as sns
from kiblib.utils.db import DbConn
from kiblib.utils.evolution_stats import EvolutionActivite
from kiblib.utils import lucas_sns_params
import warnings
warnings.filterwarnings("ignore")

In [2]:
db_conn = DbConn().create_engine()

query = """SELECT i.biblionumber,i.barcode,i.ccode,i.dateaccessioned,i.location,i.itemcallnumber,i.homebranch,i.itype, b.itemtype 
FROM koha_prod.items i
LEFT JOIN biblioitems b ON b.biblionumber = i.biblionumber
WHERE YEAR(dateaccessioned)='2024' 
AND homebranch = 'MED'
AND LOCATION != 'MED0A'"""

In [3]:
ref_acquereurs = pd.read_excel("/home/kibini/kibini2/referentiels/ref_acquereurs.xlsx")

In [4]:
ref_acquereurs

,ccode,lib,responsable_collection,lib_public
0,AAPATAP,AAP - Arts plastiques,Christine,Arts plastiques
1,AAPATLC,AAP - Loisirs créatifs,Christine,Loisirs créatifs
2,AAPBDCM,AAP - Bande dessinée : Comics,Hervé,Bande dessinée : Comics
3,AAPBDDC,AAP - Bande dessinée : documentaires,Hervé,Bande dessinée (documentaires)
4,AAPBDER,AAP - Bande dessinée érotique,Hervé,Bande dessinée érotique
...,...,...,...,...
144,PPIPIZZ,PPI - Patrimoine iconographique,Stéphanie,Patrimoine iconographique
145,PRRFIZZ,PRR - Films autour de Roubaix et sa région,Christine,Films autour de Roubaix et sa région
146,PRRMEZZ,PRR - FLRS de prêt,hervé,Fonds local régional sonore empruntable
147,PRRRGZZ,PRR - Région,Stéphanie,Région


In [5]:
acquisitions_2024 = pd.read_sql(query,con=db_conn)

In [6]:
acquisitions_2024

,biblionumber,barcode,ccode,dateaccessioned,location,itemcallnumber,homebranch,itype,itemtype
0,362060,C2600009312,JBDMGZZ,2024-01-02,MED2A,E BD/MANGAS,MED,PRETLIV,None
1,362057,C2600009313,ALTTLZZ,2024-01-02,MED1A,LIT/PUJ,MED,PRETLIV,None
2,362053,C2600009314,JBDMGZZ,2024-01-02,MED2A,E BD/MANGAS,MED,PRETLIV,None
3,362052,C2600009315,JBDMGZZ,2024-01-02,MED2A,E BD/MANGAS,MED,PRETLIV,None
4,362055,C2600009043,JBDZZZZ,2024-01-02,MED2A,E BD/GIL,MED,PRETLIV,None
...,...,...,...,...,...,...,...,...,...
11648,277125,C2700000311,JBDSRZZ,2024-12-27,MED2A,E BD/SERIES,MED,PRETLIV,None
11649,371474,C2500038280,JABEFAE,2024-12-28,MED2A,E A/VEL,MED,PRETLIV,None
11650,371051,C2400005660,ACFLALE,2024-12-28,MED3A,ALL ML,MED,PRETLIV,None
11651,371050,C2400005661,ACFLALE,2024-12-28,MED3A,ALL ML,MED,PRETLIV,None


In [7]:
acquisitions_2024_info_acq = acquisitions_2024.merge(ref_acquereurs,left_on='ccode',right_on='ccode',how='left')

In [8]:
acquisitions_2024_info_acq['nb'] = 1

In [9]:
acquisitions_2024_info_acq.groupby(['responsable_collection'])['nb'].sum().to_frame('Nb acquisitions')

,Nb acquisitions
responsable_collection,
Anne-Sophie,776
Chantal,2341
Christine,1218
Hervé,1355
Laetitia C,64
Mathilde,1604
Maïté,83
Pascale,1450
Stéphanie,2063


In [10]:
query2 = """SELECT av.authorised_value ,av.lib AS 'type de support'
FROM koha_prod.authorised_values av 
WHERE category = 'SUGGEST_FORMAT'"""

lib = pd.read_sql(query2,con=db_conn)

In [11]:
lib

,authorised_value,type de support
0,PRETBD,Prêt de bandes dessinées
1,PRETCAR,Prêt de cartes routières
2,PRETDVD,Prêt de DVD
3,PRETLIV,Prêt d'imprimés
4,PRETLS,Prêt de livres audio
5,PRETMAT,"Prêt de matériel (casque audio, clé USB, ...)"
6,PRETMUL,Prêt de documents multimédia
7,PRETPAR,Prêt de partitions
8,PRETPER,Prêt de périodiques
9,PRETSON,Prêt de documents sonores


In [12]:
acquisitions_2024_info_acq = acquisitions_2024_info_acq.merge(lib,left_on='itype',right_on='authorised_value',how='left')

In [13]:
acquisitions_2024_info_acq.groupby(['responsable_collection',
                                    'type de support'
                                   ])['nb'].sum().to_frame('Nb acquisitions 2024')

Nb acquisitions 2024
responsable_collection type de support                                
Anne-Sophie            Prêt d'imprimés                             595
                       Prêt de DVD                                   6
                       Prêt de périodiques                         175
Chantal                Prêt d'imprimés                            2341
Christine              Prêt d'imprimés                             549
                       Prêt de DVD                                 373
                       Prêt de périodiques                         296
Hervé                  Prêt d'imprimés                             708
                       Prêt de DVD                                  29
                       Prêt de documents sonores                   500
                       Prêt de périodiques                         118
Laetitia C             Prêt d'imprimés                              42
                       Prêt de périodiques                          22
Mathilde               Prêt d'imprimés                            1504
                       Prêt de livres audio                         46
                       Prêt de périodiques                          54
Maïté                  Prêt d'imprimés                              27
                       Prêt de périodiques                          56
Pascale                Prêt d'imprimés                            1341
                       Prêt de DVD                                  75
                       Prêt de documents sonores                     1
                       Prêt de périodiques                          33
Stéphanie              Prêt d'imprimés                             637
                       Prêt de documents sonores                   957
                       Prêt de périodiques                         469
Zina                   Prêt d'imprimés                             149
                       Prêt de périodiques                          64

In [18]:
acquisitions_2024_info_acq[(acquisitions_2024_info_acq['responsable_collection']=='Stéphanie') &
                           (acquisitions_2024_info_acq['type de support']=="Prêt de documents sonores")
                          ].groupby('lib').size()

lib
PEN - Fonds Charles Verstraete    957
dtype: int64